In [8]:
import os
import sys
import numpy as np
notebook_dir = os.path.abspath(os.path.dirname(''))
project_root = os.path.dirname(notebook_dir)
sys.path.append(project_root)
import pandas as pd
from src.utils.db_utils import get_connection, execute_query

season = 2024
stat = "turnovers"

def offensive_mettrics(season, stat):
    
    query=f"""SELECT *
    FROM stats.teamstats
    WHERE season = {season}
    """
    query_results=execute_query(query)

    weeks=np.unique(query_results.loc[:,"week"].values)
    teams=np.unique(query_results.loc[:,"teamid"].values)

    stats_pg_df=pd.DataFrame(index=weeks, columns=teams)

    for i_week in weeks:
        current_week_idx=np.where(query_results["week"]==i_week)[0]
        current_week_df=pd.DataFrame(query_results.iloc[current_week_idx]).set_index('teamid')
        stats_pg_df.loc[i_week, current_week_df.index]=current_week_df.loc[:, stat]
        
    average_stats_pg_df=pd.DataFrame(index=weeks, columns=teams)
    
    for i_week in range(weeks.shape[0]):
        for i_team in teams:
            current_week=weeks[i_week]
            average_stats_pg_df.loc[current_week,i_team]=np.mean(stats_pg_df.loc[:current_week, i_team].dropna())
        
    return stats_pg_df, average_stats_pg_df





# Test the function

stats_pg_df, average_stats_pg_df = offensive_mettrics(season, stat)

print("Yards Allowed Per Game:")
print(stats_pg_df)

print("\nAverage Yards Allowed:")
print(average_stats_pg_df)


Successfully connected to the database!
Yards Allowed Per Game:
    ARI  ATL  BAL  BUF  CAR  CHI  CIN  CLE  DAL  DEN  ...  NWE  NYG  NYJ  PHI  \
1     1    3    1    1    3    1    2    2    0    3  ...    0    2    2    3   
2     1    0    1    0    1    2    1    0    2    2  ...    0    1    0    1   
3     1    1    0    0    0    3    0    2    1    1  ...    1    2    0    2   
4     1    1    1    1    1    0    1    1    0    1  ...    3    1    1    2   
5     1    1    1    0    3    0    1    1    3    0  ...    0    1    3  NaN   
6     3    1    1    0    2    1    1    0    5    2  ...    4    1    1    0   
7     1    3    1    0    2  NaN    0    2  NaN    1  ...    0    0    2    0   
8     0    0    0    1    2    1    2    1    2    2  ...    0    2    0    0   
9     2    1    0    1    1    0    1    3    0    1  ...    3    1    1    1   
10    0    1    0    2    1    0    1  NaN    5    0  ...    1    3    1    2   
11  NaN    1    3    1  NaN    0    0    0   